## Notebook 3 of 5 — Run the NPS Water Balance Model

Runs the NPS WBM (`wbm.run_nps_wbm_points`) at a daily timestep using the gridMET climate from notebook 2, aggregates to monthly, plots monthly AET, and checks for unit/sign issues (negative or missing AET). Results are cached to `Data/gridmet_cache/wbm_results_*.csv`.

*Part of a 5-notebook pipeline (run in order; each caches its outputs to `Data/` so later notebooks can be re-run without repeating expensive GEE/download steps): `01_select_parks_towers` -> `02_load_flux_openet_gridmet` -> `03_run_wbm` -> `04_validate_and_calibrate` -> `05_figures_and_export`.*

<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2025/blob/main/lectures/lecture4-ET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CIVE 523 Final Project – Kristen Cognac, February 27, 2026

Objective: The National Park Service (NPS) Water Balance Model (WBM) employs a set of parameterizations and functions to estimate a daily water balance for components rain, snow, snowmelt, soil water storage, evapotranspiration, and lumped runoff (surface water and groundwater). Actual evapotranspiration (ETa) is estimated using a simple “bucket-type” approach wherein potential evapotranspiration (Oudin, 2005) is maximized to the extent of available soil moisture storage. The NPS WBM has been recently applied across parks to assess past and future changes in water availability (e.g., Thoma, 2019; Thoma, 2020). However, detailed validation of model components has yet to be conducted. 

OpenET, with daily estimates of ETa from six remote sensing models, provides a robust, spatially continuous dataset for validating NPS WBM ETa. This project will evaluate NPS WBM ETa accuracy by comparing it to OpenET monthly and annual timeseries using common statistical metrics. Given that OpenET has errors too, OpenET accuracy will also be assessed using ground-truth measurements from flux towers. The implications of ETa inaccuracy on water availability assessments will be considered by comparing error magnitude to the total water balance at each location. This project will rely heavily on previous data and analysis compiled by Volk et al. (2024), which assessed the accuracy of OpenET across CONUS. In particular, they provide corrected timeseries of in situ ETa (and OpenET) through Zenodo repositories that are useful for estimating OpenET accuracy (Volk et al., 2023a; 2023b).

Methods: Six US National Park sites within 20 km of eddy covariance towers (AmeriFlux, USGS NWSC) are selected for comparison. Towers and park elevations range from 0 to 4,350m, enabling assessment across an elevation gradient. Daily ETa (1999-2024) will be calculated for each park centroid (4km GridMET) using the NPS WBM. The single grid cell minimizes errors from variable park sizes and limits data processing. (Note: Generating daily NPS WBM ETa requires downloading GridMET timeseries, extracting parameters (soil water capacity, slope, and elevation), and running WBM functions). The aerial analysis extent is 80 km2 distributed across the five parks. OpenET (30m grid) will be spatially averaged for each 4km2 cell to generate daily, monthly, and annual Eta. NPS WBM ETa accuracy will be assessed using: linear regression slope (forced through the origin), mean bias error (MBE), mean absolute error (MAE), root-mean-square error (RMSE), and coefficient of determination (r2). Flux tower data (Volk et al., 2023a) will be resampled to daily, monthly, and annual values and compared to the corresponding OpenET ETa pixel (Volk et al., 2023b). OpenET accuracy will be similarly assessed using linear regression, MBE, MAE, RMSE, and r2). Relative error metrics (e.g., MAE/Precipitation) will be evaluated to understand impacts on water availability assessments.

References:

Oudin, L., Hervieu, F., Michel, C., Perrin, C., Andréassian, V., Anctil, F., & Loumagne, C. (2005). Which potential evapotranspiration input for a lumped rainfall–runoff model?: Part 2—Towards a simple and efficient potential evapotranspiration model for rainfall–runoff modelling. Journal of hydrology, 303(1-4), 290-306.

Thoma, D. P., Munson, S. M., & Witwicki, D. L. (2019). Landscape pivot points and responses to water balance in national parks of the southwest US. Journal of Applied Ecology, 56(1), 157-167.

Thoma, D. P., Tercek, M. T., Schweiger, E. W., Munson, S. M., Gross, J. E., & Olliff, S. T. (2020). Water balance as an indicator of natural resource condition: Case studies from Great Sand Dunes National Park and Preserve. Global Ecology and Conservation, 24, e01300.

John M. Volk, Justin L. Huntington, Forrest Melton, Blake Minor, Tianxin Wang, Saseendran S. Anapalli, Raymond G. Anderson, Steven R. Evett, Andrew N. French, Richard Jasoni, Nicolas Bambach, William P. Kustas, Joseph G. Alfieri, John Prueger, Lawrence Hipps, Lynn McKee, Sebastian J. Castro Bustamante, Maria del Mar Alsina, Andrew McElrone, … Martha Anderson. (2023a). Post-processed data and graphical tools for a CONUS-wide eddy flux evapotranspiration dataset (1.0.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.7636781

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., Kilic, A., Ruhoff, A., Senay, G. B., Minor, B., Morton, C., Ott, T., Johnson, L., Andrade, B. C. D., Carrara, W., Doherty, C. T., Dunkerly, C., Friedrichs, M., Guzman, A., … Yang, Y. (2023b). OpenET model data for assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications [Data set]. Zenodo. https://doi.org/10.5281/zenodo.10119477

Volk, J. M., Huntington, J. L., Melton, F. S., Allen, R., Anderson, M., Fisher, J. B., ... & Yang, Y. (2024). Assessing the accuracy of OpenET satellite-based evapotranspiration data to support water resource and land management applications. Nature Water, 2(2), 193-205.





# Setup Workspace

Import necessary libraries.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMPORTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Make the repo root (parent of notebooks/) importable ─────────────────────
import sys
from pathlib import Path as _Path
_REPO_ROOT = _Path.cwd().parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# ── Standard library ──────────────────────────────────────────────────────────
import importlib
import os
import time
import tempfile
import warnings
import zipfile
from datetime import date
from pathlib import Path

# ── Scientific computing ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr

# ── Geospatial ────────────────────────────────────────────────────────────────
import geopandas as gpd
import rasterio
import shapely
from rasterio.transform import rowcol
from shapely.geometry import Point
from adjustText import adjust_text

# ── Google Earth Engine ───────────────────────────────────────────────────────
import ee
import geemap

# ── Visualization ─────────────────────────────────────────────────────────────
import branca.colormap as cm
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# ── Statistics & optimization ─────────────────────────────────────────────────
import requests
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import pdist, squareform

# ── Utilities ─────────────────────────────────────────────────────────────────
from adjustText import adjust_text
from tqdm import tqdm

# ── NPS WBM (local package, repo_root/wbm/) ───────────────────────────────────
import wbm
from wbm import (
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run
)


Authenticate Google Earth Engine

In [ ]:
#if not ee.data._credentials:
ee.Authenticate()
ee.Initialize(project='modis-475315')

Define helper functions

In [ ]:
# functions needed for this workflow

# this function is used to add a google earth engine layer to an existing folium map,
# for visualization purposes. Folium is a python package that can put rasters/shapefiles on a basemap
# the function below is run using an existing folium map. If the folium map defines is my_map, then
# my_map.add_ee_layer(ee_object,name)
# where ee_object is the object defined in google earth engine, and name is the label in folium
def add_ee_layer(self, ee_object, name):
    try:
        # display ee.Image()
        if isinstance(ee_object, ee.image.Image):
            range = ee.Image(ee_object).reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
            vals = range.getInfo()
            min=list(vals.items())[0][1]
            max=list(vals.items())[1][1]
            vis = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}

            map_id_dict = ee.Image(ee_object).getMapId(vis)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
            colormap = cm.LinearColormap(vmin=min,vmax=max,colors=['blue', 'white','red']).to_step(n=10)
            colormap.caption=name
            self.add_child(colormap)
        # display ee.ImageCollection()
        elif isinstance(ee_object, ee.imagecollection.ImageCollection):
            ee_object_new = ee_object.mosaic()
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
        # display ee.Geometry()
        elif isinstance(ee_object, ee.geometry.Geometry):
            folium.GeoJson(
            data = ee_object.getInfo(),
            name = name,
            overlay = True,
            control = True
        ).add_to(self)
        # display ee.FeatureCollection()
        elif isinstance(ee_object, ee.featurecollection.FeatureCollection):
            ee_object_new = ee.Image().paint(ee_object, 0, 2)
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
        ).add_to(self)

    except Exception as e:
        print("Could not display {}".format(name))
        print(e)


# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  min=list(vals.items())[0][1]
  max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
 # range = img.reduceRegion(ee.Reducer.percentile([1, 99]),scale=10000)
 # vals = range.getInfo()
 # min=list(vals.items())[0][1]
 # max=list(vals.items())[1][1]
 # visParams = {'min': min, 'max': max, 'palette': ['0000FF', 'FFFFFF','FF0000']}
  return(img)

# load prism data
def get_prism_image(date1,date2,geometry):

  prism = ee.ImageCollection('OREGONSTATE/PRISM/AN81m')
  prism_img = prism.filterDate(date1,date2).select('ppt').mean().clip(geometry)
  return(prism_img) # returns prism average monthly precipitation, in mm

# load landsat 8 data
def get_l8_image(date1,date2,geometry):

  l8 = ee.ImageCollection('LANDSAT/LC08/C01/T1_RT')
  l8_img = l8.filterDate(date1,date2).mean().clip(geometry)
  return(l8_img)

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

# to create an elevation raster from the USGS NED in google earth engine from a user-defined geometry
def get_elev(geometry):

  elev = ee.Image('USGS/NED').clip(geometry)
  return(elev)

# to create an elevation raster from the SRTM in google earth engine from a user-defined geometry
def get_srtm(geometry):

  elev = ee.Image('USGS/SRTMGL1_003').clip(geometry)
  return(elev)

# to create a temporally averaged precipitation raster from GPM from a user-defined geometry
def get_gpm_image(date1,date2,geometry):

  gpm = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07')
  gpm_img = gpm.filterDate(date1,date2).select('precipitation').mean().multiply(24*365/12).clip(geometry) # convert from mm/hour to mm/month
  return(gpm_img) # returns gpm average monthly precipitation in mm

# to create a temporally averaged actual ET raster from the openET ensemble from a user-defined geometry
def get_openET_image(date1,date2,geometry):

  openET = ee.ImageCollection('OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0')
  openET_img = openET.filterDate(date1,date2).select('et_ensemble_mad').mean().clip(geometry)
  return(openET_img)

# to create a temporally averaged reference ET raster from the openET ensemble from a user-defined geometry
def get_RET(date1,date2,geometry):

  ETR = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
  ETR_image = ETR.filterDate(date1,date2).select('etr').mean().multiply(365/12).clip(geometry) # convert from mm/day to mm/month
  return(ETR_image)

# load sentinel 2 data
def get_s2_image(date1,date2,geometry):

    s2 = ee.ImageCollection('COPERNICUS/S2')
    s2_img = s2.filterDate(date1,date2).filterBounds(geometry).first().clip(geometry)
    return(s2_img)

# Add EE drawing method to folium (not a function)
folium.Map.add_ee_layer = add_ee_layer

def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=1000,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):
  """Creates a region reduction function.

  Creates a region reduction function intended to be used as the input function
  to ee.ImageCollection.map() for reducing pixels intersecting a provided region
  to a statistic for each image in a collection. See ee.Image.reduceRegion()
  documentation for more details.

  Args:
    geometry:
      An ee.Geometry that defines the region over which to reduce data.
    reducer:
      Optional; An ee.Reducer that defines the reduction method.
    scale:
      Optional; A number that defines the nominal scale in meters of the
      projection to work in.
    crs:
      Optional; An ee.Projection or EPSG string ('EPSG:5070') that defines
      the projection to work in.
    bestEffort:
      Optional; A Boolean indicator for whether to use a larger scale if the
      geometry contains too many pixels at the given scale for the operation
      to succeed.
    maxPixels:
      Optional; A number specifying the maximum number of pixels to reduce.
    tileScale:
      Optional; A number representing the scaling factor used to reduce
      aggregation tile size; using a larger tileScale (e.g. 2 or 4) may enable
      computations that run out of memory with the default.

  Returns:
    A function that accepts an ee.Image and reduces it by region, according to
    the provided arguments.
  """

  def reduce_region_function(img):
    """Applies the ee.Image.reduceRegion() method.

    Args:
      img:
        An ee.Image to reduce to a statistic by region.

    Returns:
      An ee.Feature that contains properties representing the image region
      reduction results per band and the image timestamp formatted as
      milliseconds from Unix epoch (included to enable time series plotting).
    """

    stat = img.reduceRegion(
        reducer=reducer,
        geometry=geometry,
        scale=scale,
        crs=crs,
        bestEffort=bestEffort,
        maxPixels=maxPixels,
        tileScale=tileScale)

    return ee.Feature(geometry, stat).set({'millis': img.date().millis()})
  return reduce_region_function

# Define a function to transfer feature properties to a dictionary.
def fc_to_dict(fc):
  prop_names = fc.first().propertyNames()
  prop_lists = fc.reduceColumns(
      reducer=ee.Reducer.toList().repeat(prop_names.size()),
      selectors=prop_names).get('list')

  return ee.Dictionary.fromLists(prop_names, prop_lists)

# generate data frame from image collection
def gee_zonal_mean_img_coll(imageCollection,geometry,scale=1000):
    reduce_iC = create_reduce_region_function(geometry = geometry, scale=scale)
    stat_fc = ee.FeatureCollection(imageCollection.map(reduce_iC)).filter(ee.Filter.notNull(imageCollection.first().bandNames()))
    fc_dict = fc_to_dict(stat_fc).getInfo()

    df = pd.DataFrame(fc_dict)
    df['date'] = pd.to_datetime(df['millis'],unit='ms')
    return(df)

def gee_zonal_mean(date1,date2,geometry,collection_name,band_name,scale=1000):
     imcol = ee.ImageCollection(collection_name).select(band_name).filterDate(date1,date2)
     df = gee_zonal_mean_img_coll(imcol,geometry,scale=scale)
     return(df)

# Convert shapefile to EE object
def shapefile_to_ee(filepath):
    # 1. Read the shapefile
    gdf = gpd.read_file(filepath)
    
    # 2. Ensure it is in WGS84 (required by Earth Engine)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    
    # 3. Convert the geometry to a GeoJSON-like mapping
    # This handles Points, Polygons, and MultiPolygons
    geojson = gdf.__geo_interface__
    
    # 4. Create an ee.FeatureCollection from the GeoJSON
    # You can then get the geometry from the collection
    ee_object = ee.FeatureCollection(geojson)
    
    return ee_object.geometry()


# read in shapefile of National Parks
def get_park_system(save: bool = False, path: str | None = None) -> gpd.GeoDataFrame:
    """Import national park boundary shapefile.

    Downloads NPS park boundary shapefiles via the Data Store REST API.

    Reference:
        https://irmaservices.nps.gov/datastore/v6/documentation
        https://irma.nps.gov/DataStore/Reference/Profile/2224545?lnv=True

    Last updated: September 2025

    Args:
        save: If True, write the result to disk as a GeoPackage.
        path: Directory to save the file. Required when save=True.

    Returns:
        GeoDataFrame of all NPS park boundaries (EPSG:4326).
    """
    download_link = "https://irma.nps.gov/DataStore/DownloadFile/733895"

    # ── Download zip to a temp file ────────────────────────────────────────
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        zip_path = tmp_dir / "boundary.zip"
        extract_dir = tmp_dir / "extracted"

        response = requests.get(download_link, stream=True, timeout=120)
        response.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        # ── Unzip and read shapefile ───────────────────────────────────────
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        shp_path = extract_dir / "nps_boundary.shp"
        parks = gpd.read_file(shp_path)

        # ── Fix any invalid geometries (mirrors st_make_valid) ─────────────
        parks["geometry"] = parks["geometry"].make_valid()

        # ── Optionally save as GeoPackage ──────────────────────────────────
        if save and path is not None:
            out_path = (
                Path(path)
                / f"nps_park_boundaries_{date.today()}.gpkg"
            )
            parks.to_file(out_path, driver="GPKG")

    return parks


setup for NPS WBM

In [ ]:
# ── Import the WBM model from the `wbm` package (repo_root/wbm/) ────────────
# All functions are also available via the `wbm` namespace, e.g. wbm.nps_wbm(...)
import importlib
import wbm

# Reload in case you edited the package mid-session without restarting the kernel
importlib.reload(wbm)

from wbm import (
    # ── Raster utilities ──────────────────────────────────────────────────
    load_wbm_rasters,       # load & cache DEM / soil / Jennings rasters
    extract_point_params,   # sample site params at point locations
    run_pipeline,           # one-call convenience: load → extract → run

    # ── Core model ───────────────────────────────────────────────────────
    nps_wbm,                # single-point daily water balance driver
    run_nps_wbm_points,     # multi-point / multi-GCM wrapper

    # ── Component functions (available if needed for custom workflows) ────
    get_freeze,             # rain/snow partitioning factor
    get_rain, get_snow,     # rainfall and snowfall
    get_melt,               # Hock degree-day snowmelt
    get_snowpack,           # snowpack accumulation
    get_ablation,           # snow sublimation / vapor loss
    get_soil,               # soil water content
    get_d_soil,             # daily change in SWC
    get_aet,                # actual evapotranspiration
    get_storage,            # linear storage reservoir (CSU addition)
    get_oudin_pet,          # Oudin PET with topographic heat-load
    get_hamon_pet,          # Hamon PET
    get_penman_monteith_pet,# FAO-56 Penman-Monteith PET
    get_daylength,          # astronomical daylength (hours)
    get_gdd,                # growing degree days
    get_deficit,            # climatic water deficit (PET − AET)

    # ── Low-level helpers (rarely called directly) ────────────────────────
    get_svp, actual_vp, atm_press, psyc_constant,
    vapor_curve, clear_sky_rad, outgoing_rad,
)

print(f"WBM functions loaded from: {_REPO_ROOT / 'wbm'}")


Load raster datasets used for the NPS WBM

In [ ]:

# Reads the three required GeoTIFFs from `Data/wbm_rasters/`, derives slope
# and aspect from the DEM (Horn's 8-neighbour method, matching
# `terra::terrain(neighbors = 8)`), and caches everything in memory.
#
# **Only needs to run once per session.**  After this, `extract_point_params()`
# works without any path arguments.
#
# | File | Content | Units |
# |---|---|---|
# | `elevation_cropped.tif` | DEM | metres |
# | `water_storage.tif` | Soil water storage capacity | cm (auto-converted → mm) |
# | `merged_jennings2.tif` | Jennings temperature climatology | °C |

# %%
# ── Path to raster directory ──────────────────────────────────────────────────
# Adjust if your rasters live elsewhere.
RASTER_DIR = "../Data/wbm_rasters"

# ── Target CRS ────────────────────────────────────────────────────────────────
# Set to None to keep the native CRS of the rasters (EPSG:4326 for NPS data).
# Set to an EPSG string (e.g. "EPSG:26913") to reproject all rasters before
# sampling — useful when your point coordinates are in a projected CRS.
TARGET_CRS = "EPSG:4326"

try:
    load_wbm_rasters(raster_dir=RASTER_DIR, target_crs=TARGET_CRS)
except ImportError as e:
    print(f"⚠️  Rasters not loaded: {e}")
    print("   Install rasterio and re-run this cell before calling extract_point_params().")
except FileNotFoundError as e:
    print(f"⚠️  Raster file not found:\n   {e}")
    print(f"   Check that RASTER_DIR = '{RASTER_DIR}' is correct.")

Define global model settings for NPS WBM

In [ ]:
# Set default WBM run parameters here.  These are passed through to
# `nps_wbm()` / `run_nps_wbm_points()` / `run_pipeline()` in later cells.
# Override any of them at call-time as needed.


# ── PET method ────────────────────────────────────────────────────────────────
# One of: "Oudin"  (default, temperature-based, topographic heat-load adjusted)
#         "Hamon"  (temperature + daylength)
#         "Penman-Monteith"  (requires tmax, tmin, and ideally RH / wind data)
PET_METHOD = "Oudin"

# ── Snowmelt ──────────────────────────────────────────────────────────────────
# Hock (2003) degree-day melt factor (mm °C⁻¹ day⁻¹).
# Hock reports ~2.5 for Gooseberry Creek, UT; NPS default is 4.
HOCK_COEF = 4.0

# ── Initial conditions ────────────────────────────────────────────────────────
SNOWPACK_INIT = 0.0   # mm SWE
SOIL_INIT     = 0.0   # mm

# ── CSU additions ─────────────────────────────────────────────────────────────
DIRECT_FRAC  = 0.0    # fraction of rainfall routed directly to runoff (0–1)
RETURN_RATE  = 1.0    # fraction of storage reservoir released per day (0–1]
PET_MULT     = 1    # multiplicative PET bias correction
SOIL_MULT    = 1    # multiplicative adjustment to SWC_Max

# ── Misc ──────────────────────────────────────────────────────────────────────
SHADE_COEFF  = 1.0    # canopy shading coefficient for Oudin PET (0–1)
T_BASE       = 0.0    # base temperature for growing degree days (°C)
TO_INCHES    = True   # True → output fluxes in inches; False → mm

print("✓ Model settings configured:")
print(f"  PET method   : {PET_METHOD}")
print(f"  Hock coef    : {HOCK_COEF} mm °C⁻¹ day⁻¹")
print(f"  Direct frac  : {DIRECT_FRAC}")
print(f"  Return rate  : {RETURN_RATE}")
print(f"  PET mult     : {PET_MULT}   |  Soil mult: {SOIL_MULT}")
print(f"  Output units : {'inches' if TO_INCHES else 'mm'}")


In [ ]:
# Verify setup (quick sanity check)
#
# Runs the model for one synthetic year at a single point to confirm the full
# stack (imports → raster cache → model) is working before you connect real
# climate data.

# %%
rng = np.random.default_rng(0)
_n  = 365
_dates = pd.date_range("2000-01-01", periods=_n)
_doy   = _dates.dayofyear.to_numpy(float)

_test_climate = pd.DataFrame({
    "date":    _dates,
    "x":       -105.5,          # lon — update to match your study area
    "y":        40.0,           # lat
    "ppt_mm":  rng.exponential(3.0, _n),
    "tmean_C": 8 * np.sin(2 * np.pi * (_doy - 80) / 365) + 5 + rng.normal(0, 2, _n),
    "GCM":     "sanity_check",
})

# Use hard-coded params so the test doesn't depend on the rasters being loaded
_test_params = {"Elev": 2400, "Slope": 10, "Aspect": 180,
                "SWC_Max": 150, "J_Temp": 1.5}

_test_result = nps_wbm(
    _test_climate, _test_params,
    pet_method    = PET_METHOD,
    hock_coef     = HOCK_COEF,
    direct_frac   = DIRECT_FRAC,
    return_rate   = RETURN_RATE,
    pet_mult      = PET_MULT,
    soil_mult     = SOIL_MULT,
    shade_coeff   = SHADE_COEFF,
    t_base        = T_BASE,
    to_inches     = False,          # mm for the sanity check
)

_unit = "mm"
print("✓ Sanity check passed — annual water balance totals:")
print(f"  {'Variable':<14}  {'Annual total':>14}")
print(f"  {'-'*30}")
for _col in ["ppt_mm", "RAIN", "SNOW", "MELT", "AET", "RUNOFF", "D"]:
    print(f"  {_col:<14}  {_test_result[_col].sum():>12.1f} {_unit}")

print(f"\n  Peak snowpack : {_test_result['PACK'].max():.1f} {_unit}")
print(f"  Max soil SWC  : {_test_result['SOIL'].max():.1f} {_unit}")
print(f"\n✓ Setup complete — ready to run NPS WBM.\n")

In [ ]:
# NPS WBM Quick-reference: key function signatures
#
# ```python
# # ── Extract site params at your points from the loaded rasters ────────────
# point_params_df = extract_point_params(points_df)
# # points_df needs columns: x (lon), y (lat)
# # Returns:  Elev, Slope, Aspect, SWC_Max, J_Temp  added to points_df
#
# # ── Run for a single point ────────────────────────────────────────────────
# result = nps_wbm(
#     daily_df     = climate_df,        # date, x, y, ppt_mm, tmean_C [, GCM]
#     point_params = point_params_df.iloc[0].to_dict(),
#     pet_method   = PET_METHOD,
#     **{k: v for k, v in globals().items()
#        if k in ("hock_coef","direct_frac","return_rate","pet_mult",
#                 "soil_mult","shade_coeff","t_base","to_inches")},
# )
#
# # ── Run for multiple points / GCMs ───────────────────────────────────────
# results = run_nps_wbm_points(
#     climate_data    = climate_df,
#     point_params_df = point_params_df,
#     pet_method      = PET_METHOD,
#     aggregate       = True,           # False → keep per-point rows
#     ret_final_cond  = False,          # True → chain into next time period
# )
#
# # ── One-call pipeline (load → extract → run) ─────────────────────────────
# results = run_pipeline(
#     climate_data = climate_df,
#     points_df    = points_df,
#     raster_dir   = RASTER_DIR,
#     pet_method   = PET_METHOD,
# )
# ```

*Loading prerequisites saved by notebooks 01-02 — needed if you're starting this notebook in a fresh kernel.*

In [ ]:
# ── Load prerequisites from notebooks 01-02 (fresh kernel = these aren't in memory) ──
flux_towers = pd.read_csv("../Data/geo_data/flux_towers.csv")
climate_gee = pd.read_csv("../Data/gridmet_cache/climate_gee_flux_towers_2016_2023.csv",
                          parse_dates=["date"])
print(f"Loaded flux_towers ({len(flux_towers)} sites) and climate_gee {climate_gee.shape} "
      f"from notebooks 01-02's cache.")


Now, we'll run the NPS WBM.

In [ ]:
read_prev = False   # Set to False to skip reading and reprocessing GEE climate data

if read_prev:
    # import instead of run
    wbm_results = pd.read_csv(
        "../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
        parse_dates=["date"]
    )
else:

    # ── Run NPS WBM for each flux tower site ─────────────────────────────────────

    # Step 1: Extract site parameters from rasters at each tower location
    # (requires load_wbm_rasters() to have been run in the setup notebook)
    print("Extracting site parameters from rasters...")
    point_params_df = extract_point_params(flux_towers)
    print(point_params_df[["site", "Elev", "Slope", "Aspect", "SWC_Max", "J_Temp"]]
        .to_string(index=False))

    # Step 2: Loop over sites and run WBM
    print("\nRunning NPS WBM...")
    wbm_frames = []

    for row in point_params_df.itertuples():

        # Subset climate to this site
        site_climate = climate_gee[climate_gee["site"] == row.site].copy()

        if site_climate.empty:
            print(f"  ⚠️  No climate data for {row.site} — skipping")
            continue

        print(f"  {row.site:<10} ({len(site_climate):,} days) ...", end=" ")

        try:
            result = nps_wbm(
                daily_df     = site_climate,
                point_params = {
                    "Elev":    row.Elev,
                    "Slope":   row.Slope,
                    "Aspect":  row.Aspect,
                    "SWC_Max": row.SWC_Max,
                    "J_Temp":  row.J_Temp,
                },
                pet_method    = PET_METHOD,
                hock_coef     = HOCK_COEF,
                direct_frac   = DIRECT_FRAC,
                return_rate   = RETURN_RATE,
                pet_mult      = PET_MULT,
                soil_mult     = SOIL_MULT,
                shade_coeff   = SHADE_COEFF,
                t_base        = T_BASE,
                to_inches     = TO_INCHES,
            )
            result["site"]      = row.site
            result["ecosystem"] = row.ecosystem
            result["state"]     = row.state
            wbm_frames.append(result)
            print("✓")

        except Exception as e:
            print(f"\n    ⚠️  {row.site} failed: {e}")

    # Step 3: Combine all sites into one DataFrame
    wbm_results = (
        pd.concat(wbm_frames, ignore_index=True)
        .sort_values(["site", "date"])
        .reset_index(drop=True)
    )

    print(f"\nWBM results shape: {wbm_results.shape}")
    print(f"Sites in output  : {sorted(wbm_results['site'].unique())}")
    print(wbm_results.head(8).to_string(index=False))

    # ── Save ──────────────────────────────────────────────────────────────────────
    wbm_results.to_csv("../Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv",
            index=False)
    print("\nSaved to Data/gridmet_cache/wbm_results_flux_towers_2016_2023.csv")

The NPS WBM is run at a daily timestep. Here we aggreagate to monthly for comparison with OpenET.

In [ ]:
# ── Summarize daily WBM results to monthly ────────────────────────────────────
#
# Aggregation rules vary by variable type:
#   Sum  → flux variables (ppt, rain, snow, melt, AET, runoff, deficit)
#   Mean → state variables (soil storage, snowpack, temperature)

# Variables to sum (fluxes — accumulate over the month)
FLUX_COLS  = ["ppt_mm", "RAIN", "SNOW", "MELT", "AET", "RUNOFF", "D",
              "etr_gridmet_mm"]

# Variables to average (states — representative value for the month)
STATE_COLS = ["SOIL", "PACK", "tmean_C"]

# Build aggregation dictionary
agg_dict = {col: "sum"  for col in FLUX_COLS  if col in wbm_results.columns}
agg_dict.update({col: "mean" for col in STATE_COLS if col in wbm_results.columns})

wbm_monthly = (
    wbm_results
    .assign(
        year  = lambda d: d["date"].dt.year,
        month = lambda d: d["date"].dt.month,
        # First day of month — makes joining with OpenET easier
        date_monthly = lambda d: pd.to_datetime(
            d["date"].dt.to_period("M").dt.to_timestamp()
        ),
    )
    .groupby(["site", "ecosystem", "state", "date_monthly", "year", "month"],
             as_index=False)
    .agg(agg_dict)
    .sort_values(["site", "date_monthly"])
    .reset_index(drop=True)
)

print(f"Monthly WBM shape : {wbm_monthly.shape}")
print(f"Sites             : {sorted(wbm_monthly['site'].unique())}")
print(f"Date range        : {wbm_monthly['date_monthly'].min().date()} → "
      f"{wbm_monthly['date_monthly'].max().date()}")
print(f"\nExpected rows     : {len(flux_towers)} sites × 96 months = "
      f"{len(flux_towers) * 96}")
print(f"Actual rows       : {len(wbm_monthly)}")

print("\nSample output:")
print(wbm_monthly.head(12).to_string(index=False))

# ── Quick sanity check: annual AET should be << annual precip ─────────────────
annual_check = (
    wbm_monthly
    .groupby("site")[["ppt_mm", "AET", "RUNOFF"]]
    .sum()
    .round(1)
)
# Convert to mm if output was in inches
if TO_INCHES:
    annual_check = (annual_check * 25.4).round(1)
    annual_check.columns = [c + "_mm" for c in annual_check.columns]

print("\nAnnual totals averaged across 2016–2023 (mm):")
print((annual_check / 8).round(1).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
wbm_monthly.to_csv(
    "../Data/gridmet_cache/wbm_monthly_flux_towers_2016_2023.csv",
    index=False
)
print("\nSaved to Data/gridmet_cache/wbm_monthly_flux_towers_2016_2023.csv")

Now plot monthly NPS WBM AET 

In [ ]:

# ── Layout ────────────────────────────────────────────────────────────────────
sites   = sorted(wbm_monthly["site"].unique())
n_sites = len(sites)
n_cols  = 5
n_rows  = int(np.ceil(n_sites / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(7, n_rows * 1.2),
    sharey=False,
    constrained_layout=True,
)
axes_flat = axes.flatten()

# ── Plot each site ────────────────────────────────────────────────────────────
for ax, site in zip(axes_flat, sites):
    sub = (
        wbm_monthly[wbm_monthly["site"] == site]
        .sort_values("date_monthly")
    )

    aet = sub["AET"] * 25.4 if TO_INCHES else sub["AET"]

    ax.plot(sub["date_monthly"], aet,
            color="steelblue", linewidth=1.5, zorder=3)
    ax.fill_between(sub["date_monthly"], aet,
                    color="steelblue", alpha=0.15)

    # Annotate with ecosystem type
    ax.annotate(sub["ecosystem"].iloc[0],
                xy=(0.04, 0.91), xycoords="axes fraction",
                fontsize=6.5, color="grey", style="italic")

    ax.set_title(site, fontsize=9, fontweight="bold", pad=4)
    ax.set_ylabel("AET (mm / month)", fontsize=7)
    ax.tick_params(axis="both", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(bottom=0)

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

# ── Hide unused axes ──────────────────────────────────────────────────────────
for ax in axes_flat[n_sites:]:
    ax.set_visible(False)

fig.suptitle(
    "Monthly NPS WBM AET — Flux Tower Sites Near National Parks",
    fontsize=12, fontweight="bold",
)

plt.savefig("../Data/open_et/wbm_aet_faceted.png", dpi=600, bbox_inches="tight")
plt.show()

The following code chunk is a check cell to see if there are negative or missing AET values and to confirm that AET is in the order of expected magnitude.

In [ ]:
# ── Issue 1: Check units — are WBM values in inches or mm? ───────────────────
print("TO_INCHES setting:", TO_INCHES)
print("\nWBM AET sample values (first 5 rows):")
print(wbm_monthly[["site", "date_monthly", "AET"]].head(10).to_string(index=False))

print(f"\nExpected range if inches : ~0.0 – 8.0 in/month")
print(f"Expected range if mm     : ~0 – 200 mm/month")
print(f"\nActual AET range         : "
      f"{wbm_monthly['AET'].min():.3f} – {wbm_monthly['AET'].max():.3f}")

# ── Issue 2: Check parameters for zero-AET sites ──────────────────────────────
zero_max_sites = ["US-Ro1", "US-Ro3", "US-Ro4", "US-Ro5", "US-Ro6", "US-xRM"]

print("\nParameters for zero-max AET sites:")
# if imported data, need to re-run
point_params_df = extract_point_params(flux_towers)
print(point_params_df[point_params_df["site"].isin(zero_max_sites)]
      [["site", "Elev", "Slope", "Aspect", "SWC_Max", "J_Temp"]]
      .to_string(index=False))

print("\nClimate summary for zero-max AET sites:")
(climate_gee[climate_gee["site"].isin(zero_max_sites)]
 .groupby("site")
 .agg(
     tmean_mean = ("tmean_C", "mean"),
     tmean_max  = ("tmean_C", "max"),
     ppt_mean   = ("ppt_mm",  "mean"),
 )
 .round(2)
 .pipe(print))

# ── Issue 3: Check negative AET sites — likely a DSOIL > W problem ────────────
neg_sites = (
    wbm_monthly[wbm_monthly["AET"] < 0]["site"]
    .unique()
    .tolist()
)
print("Sites with negative AET:", neg_sites)

if not neg_sites:
    print("No negative AET values found — issue resolved!")
else:
    print("\nDaily WBM output for first negative site (first 20 rows):")
    test_site    = neg_sites[0]
    test_climate = climate_gee[climate_gee["site"] == test_site].copy()
    test_params  = point_params_df[point_params_df["site"] == test_site].iloc[0]
    test_result = nps_wbm(
        daily_df     = test_climate,
        point_params = {
            "Elev":    test_params.Elev,
            "Slope":   test_params.Slope,
            "Aspect":  test_params.Aspect,
            "SWC_Max": test_params.SWC_Max,
            "J_Temp":  test_params.J_Temp,
            },
            pet_method = PET_METHOD,
            hock_coef  = HOCK_COEF,
            to_inches  = TO_INCHES
            )
    print(test_result[["date", "ppt_mm", "RAIN", "SNOW", "MELT",
                    "W", "PET_mod", "SOIL", "DSOIL", "AET"]]
      .head(20).to_string(index=False))
    print(f"\nAET negative rows: {(test_result['AET'] < 0).sum()} / {len(test_result)}")
    print(f"DSOIL range      : {test_result['DSOIL'].min():.4f} – "
      f"{test_result['DSOIL'].max():.4f}")
    print(f"W range          : {test_result['W'].min():.4f} – "
      f"{test_result['W'].max():.4f}")
    print(f"PET range        : {test_result['PET_mod'].min():.4f} – "
      f"{test_result['PET_mod'].max():.4f}")